In [ ]:
from langchain.agents import initialize_agent, AgentType, Tool
# from langchain.llms import Cohere
from datetime import datetime
import random
import oci
from LoadProperties import LoadProperties

properties=LoadProperties()

#  Initialize Cohere
# llm = Cohere(cohere_api_key=COHERE_API_KEY, model="command-r-plus-08-2024", temperature=0.5)

from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI

llm = ChatOCIGenAI(
      model_id='meta.llama-3.3-70b-instruct',
      service_endpoint=properties.getEndpoint(),
      compartment_id=properties.getCompartment(),auth_type='INSTANCE_PRINCIPAL',
      model_kwargs={ "max_tokens": 600},)


#  Define sample tools 
def get_time(_: str) -> str:
    return datetime.now().strftime("It's %A, %d %B %Y, %I:%M:%S %p")

def inspirational_quote(_: str) -> str:
    quotes = [
        "Stay hungry, stay foolish. – Steve Jobs",
        "Talk is cheap. Show me the code. – Linus Torvalds"
    ]
    return random.choice(quotes)

tools = [
    Tool(name="GetTime", func=get_time, description="Get the current date and time."),
    Tool(name="GetQuote", func=inspirational_quote, description="Returns an inspirational quote.")
]

# query = "What is the current time and also tell me a quote?"


### 1. AgentType.ZERO_SHOT_REACT_DESCRIPTION

In [ ]:
#  Best for: Simple tool reasoning without chat history.
agent1 = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

query = "What is the current time and also tell me a quote?"

response1 = agent1.invoke(query)
print("ZERO_SHOT_REACT_DESCRIPTION Output:\n", response1)


### 2. AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION

In [ ]:
# Best for: Needing intermediate reasoning steps or structured output.
agent2 = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)
response2 = agent2.invoke(query)
print("STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION Output:\n", response2)


## 3. AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION

In [ ]:
# Best for: Multi-turn conversations that require memory context.
query = "What is the current time and also tell me a quote?"

from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

agent3 = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)
response3 = agent3.invoke(query)
print("CHAT_CONVERSATIONAL_REACT_DESCRIPTION Output:\n", response3)
